# CHB-MIT 六通道 Detection（教师）

`colab/` 文件夹放在 MyDrive 根目录，把这个笔记本上传到 Colab，挂载后按顺序跑。

- 代码：`/content/drive/MyDrive/colab`
- EDF：`/content/chbmit`（跑完一个患者删一个）
- 结果：`/content/drive/MyDrive/colab/results/{pid}/`

教师只用 ictal(正例) / interictal(负例)，preictal 整段丢掉；六通道一起喂一个 `Detection`，每折从头训。


## 0. 挂载 Drive、导入

In [ ]:
!pip install -q mne


In [ ]:
import gc
import json
import shutil
import sys
from datetime import datetime
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

CODE = Path("/content/drive/MyDrive/colab")
DATA = Path("/content/chbmit")
OUT = CODE / "results"

sys.path.insert(0, str(CODE))
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score

import tools as t
import colab_tools as ct
from models import Detection

print("device =", t.DEVICE)

## 1. 超参

In [ ]:
ELECS = ["Fp1-F7", "Fp2-F8", "T7-P7", "T8-P8", "P7-O1", "P8-O2"]
# chb12、chb24 不要
PATIENTS = [p for p in ["chb%02d" % i for i in range(1, 25)] if p not in ("chb12", "chb24")]
# PATIENTS = ["chb01"]                # 调试时只跑一个

SPH, SOP = 2 * 60, 30 * 60           # 发作前留白 2 min，发作前期 30 min
INTER_THR = 2 * 60 * 60              # 发作间期离发作至少 2 h
WIN_LEN, WIN_OL = 4, 2               # 4 s 窗口，50% 重叠
SEED = 3
VAL = 0.1

LR = 0.001
BATCH, PATIENCE, EPOCHS = 32, 15, 150

CONFIG = dict(learning_rate=LR, batch_size=BATCH, patience=PATIENCE, epochs=EPOCHS,
              seed=SEED, val=VAL, window_length=WIN_LEN, window_overlap=WIN_OL,
              sph=SPH, sop=SOP, interictal_thr=INTER_THR)

## 2. 主流程：下载一个 → 训练 → 测试 → 存一个

跑一个患者：
1. `ct.download_patient` 下 EDF
2. `t.preprocess` 切窗打标签
3. 按电极做 z-score（mne 读出来是伏特，std 只有 1e-4，BN 的 running_var 降不到这个量级）
4. `t.teacher_train_val_test` 按 LOOCV 切成张量
5. 每折从头训一个 `Detection`，按 val loss 早停、选模型
6. 用最好权重测该折的发作，写 manifest
7. 结果存 Drive，删本地 EDF

中途挂了就把 `PATIENTS` 改一下从断点继续，已下载的 EDF 会跳过。

In [ ]:
loss_function = nn.BCEWithLogitsLoss()

for pid in PATIENTS:
    done = OUT / pid / ("%s.json" % pid)
    if done.exists():
        print("[跳过] 已完成", pid)
        continue
    print("=" * 15, pid, "=" * 15)
    out = OUT / pid
    weight_dir = out / "weight"
    weight_dir.mkdir(parents=True, exist_ok=True)

    # 1. 下载
    pdir = ct.download_patient(pid, DATA)

    # 2. 预处理 -> windows / meta
    windows, meta = t.preprocess(str(pdir), SPH, SOP, INTER_THR, WIN_LEN, WIN_OL, ELECS)

    # 3. 统一到 O(1)，BN 的初值才对得上
    for e in ELECS:
        windows[e] = ((windows[e] - windows[e].mean()) / (windows[e].std() + 1e-8)).astype(np.float32)

    # 4. 划分
    dataset = t.teacher_train_val_test(ELECS, meta, windows, SEED, (VAL, VAL))
    del windows, meta
    gc.collect()

    # 5. 每折重训
    manifest = {"config": CONFIG, "weights": [], "folds": []}
    now = datetime.now().strftime("%y%m_%d_%H%M%S")
    for fold in range(len(dataset)):
        d = dataset[fold]
        train_loader, val_loader, test_loader = [
            DataLoader(TensorDataset(d[k + "_wins"], d["y_" + k]), batch_size=BATCH, shuffle=(k == "train"))
            for k in ("train", "val", "test")]

        teacher = Detection().to(t.DEVICE)
        optimizer = torch.optim.Adam(teacher.parameters(), lr=LR)
        schedule = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=5, min_lr=1e-6)

        # 验证集每边只有三四十个窗，AUC 在 0.5 附近乱跳，按 loss 选模型
        best_val_loss, best_epoch, best_state_dict = np.inf, 0, {}
        patience_count = 0
        print("===== %s fold %d seizure %s =====" % (pid, fold + 1, d["seizure_id"]))

        for epoch in range(EPOCHS):
            teacher.train()
            train_loss = 0
            for x, y in train_loader:
                x, y = x.to(t.DEVICE), y.to(t.DEVICE)
                optimizer.zero_grad()
                loss = loss_function(teacher(x), y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            teacher.eval()
            val_loss, val_probs, val_targets = 0, [], []
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(t.DEVICE), y.to(t.DEVICE)
                    out = teacher(x)
                    val_loss += loss_function(out, y).item()
                    val_probs.extend(torch.sigmoid(out).cpu().numpy().flatten())
                    val_targets.extend(y.cpu().numpy().flatten())

            train_loss /= len(train_loader)
            val_loss /= len(val_loader)
            val_auc = roc_auc_score(val_targets, val_probs)
            schedule.step(val_auc)
            print("epoch:%d avg_loss:%.4f avg_val_loss:%.4f,val_auc:%.4f,lr:%.8f"
                  % (epoch + 1, train_loss, val_loss, val_auc, optimizer.param_groups[0]["lr"]))

            if val_loss < best_val_loss:
                best_val_loss, best_epoch, patience_count = val_loss, epoch + 1, 0
                best_state_dict = {k: v.cpu().clone() for k, v in teacher.state_dict().items()}
            else:
                patience_count += 1
                if patience_count >= PATIENCE:
                    print("Early stop！！！")
                    break

        # 用最好的权重测这一折的发作
        teacher.load_state_dict(best_state_dict)
        teacher.eval()
        test_probs, test_targets = [], []
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(t.DEVICE), y.to(t.DEVICE)
                test_probs.extend(torch.sigmoid(teacher(x)).cpu().numpy().flatten())
                test_targets.extend(y.cpu().numpy().flatten())

        test_probs = np.array(test_probs)
        test_targets = np.array(test_targets)
        auc = roc_auc_score(test_targets, test_probs)
        acc = ((test_probs > 0.5) == test_targets).mean()
        print("fold:%d n:%d epoch:%d AUC:%.4f ACC:%.4f"
              % (fold + 1, len(test_targets), best_epoch, auc, acc))

        path = weight_dir / ("%s_teacher_fold%d.pth" % (now, fold + 1))
        torch.save(best_state_dict, path)
        manifest["weights"].append(str(path))
        manifest["folds"].append({
            "fold": fold + 1,
            "seizure_id": int(d["seizure_id"]),
            "n_train": len(d["y_train"]), "n_val": len(d["y_val"]), "n_test": len(test_targets),
            "best_epoch": best_epoch, "best_val_loss": round(float(best_val_loss), 4),
            "auc": round(float(auc), 4), "acc": round(float(acc), 4),
        })

        del teacher, best_state_dict
        torch.cuda.empty_cache()

    del dataset
    torch.cuda.empty_cache()
    gc.collect()

    # 6. 存结果、删 EDF
    path = weight_dir / ("%s_teacher_manifest.json" % now)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=4, ensure_ascii=False)
    shutil.copy(path, out / ("%s.json" % pid))
    ct.clear_patient(pid, DATA)
    print("[完成]", pid, "| AUC: %.4f" % np.mean([f["auc"] for f in manifest["folds"]]))

## 3. 汇总

In [ ]:
df = ct.summarize_teacher(OUT)
print("教师 AUC: %.4f | ACC: %.4f | 折数: %d" % (df.auc.mean(), df.acc.mean(), len(df)))
df